# Amazon Sentiment Review Analysis
# Tokenization
## Sky Shealy & Abigail Ma
## 11/4/2026

Mounting Google Drive:

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Reading in data:
- For each row, there is a 0.5% chance that it is retained in the resulting data frame, which effectively gives us a simple random sample of ~0.5% of our data.

In [21]:
import random, json
import pandas as pd

file = "/content/drive/MyDrive/DSA 495 Project/Data/Video_Games.jsonl"
p = 0.005  # keep ~0.5% of rows
random.seed(495)

rows = []
with open(file, 'r') as fp:
    for line in fp:
        if random.random() < p:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
print(df.shape)

(23118, 10)


Drops unnecessary columns:

In [22]:
df = df.drop(columns=["verified_purchase", "helpful_vote", "timestamp", "user_id", "parent_asin", "asin", "images"])

Viewing Data:

In [23]:
df.head(3)

,rating,title,text
0,5.0,I am in love with this one!,My favourite character from Kirby! Looks just ...
1,4.0,Works Well,This is a decent/good controller for the Ninte...
2,4.0,Works for gaming and for business,Razor Black Widow Elite Keyboard<br /><br />I’...


## Tokenization

In [24]:
import re
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Cleaning up html line break tags from the reviews
def clean(s):
    s = re.sub(r"<br\s*/?>", " ", str(s))
    return re.sub(r"\s+", " ", s).strip()

titles = df["title"].fillna("").map(clean).tolist()
texts = df["text"].fillna("").map(clean).tolist()

# Tokenizing title & sentence pairs
enc = tokenizer(
    titles,
    texts,
    truncation="only_second",  # trim the review text & not the title
    max_length=256,
    padding = False # We can do this per batch later for more efficiency
)

# Keep the tokenized output
df["input_ids"] = enc["input_ids"]
df["token_type_ids"] = enc["token_type_ids"]
df["attention_mask"] = enc["attention_mask"]

# Check
print(tokenizer.decode(df["input_ids"][0]))
print(f"Amount of Tokens: {len(df["input_ids"][0])}")

[CLS] I am in love with this one! [SEP] My favourite character from Kirby! Looks just right!, super cute I am in love with this figurine never taking it out and looks amazing I will place it next to kirby on my stand. Arrived in perfect condition as well. [SEP]
Amount of Tokens: 62


[CLS] is a token from BERT that signals the beginning of the text.

[SEP] signals a seperator of the text. It will go between the title and the review of the text.

In [25]:
df.head(3)

,rating,title,text,input_ids,token_type_ids,attention_mask
0,5.0,I am in love with this one!,My favourite character from Kirby! Looks just ...,"[101, 146, 1821, 1107, 1567, 1114, 1142, 1141,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,4.0,Works Well,This is a decent/good controller for the Ninte...,"[101, 5853, 2119, 102, 1188, 1110, 170, 11858,...","[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,4.0,Works for gaming and for business,Razor Black Widow Elite Keyboard<br /><br />I’...,"[101, 5853, 1111, 13926, 1105, 1111, 1671, 102...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


`input_ids` is each token ID in the review. We see each one starts with 101 because that is the [CLS] token.

`token_type_ids` shows if the token is a part of the title (=0) or text (=1)

`attention_mask` shows which tokens are real (1) versus padding (0). It is all 1s for now. Later, we will apply padding so that each sequence in the same training batch is the same length. By doing this per batch, we can save computational power as opposed to doing it all at the beginning. This is because some batches will have a shorter max sequence length than others.

## Checking Review Lengths
Because we are truncating reviews at 256 tokens for computational efficiency, we want to see how many reviews are getting truncated. From the below code, about 5.8% of reviews hit the max limit. This seems acceptable. If the model struggles on long reviews, we can increase the max length before moving on with a final model.

In [26]:
lengths = pd.Series([len(x) for x in enc["input_ids"]])
print(lengths.describe())
print((lengths == 256).mean())  # fraction truncated

count    23118.000000
mean        65.402154
std         67.888753
min          5.000000
25%         18.000000
50%         40.000000
75%         82.000000
max        256.000000
dtype: float64
0.0582662860109006
